# 🌍 Climate Chat Evaluation System v2.1.1


## Quick Start:
1. in your google drive create a folder named "climet-exam"
2. Upload `knowledge_components.json` to the folder
2. Put Student dialog folder in path: ../climet-exam/student_dilaog

## 1️⃣ Setup and Imports

In [1]:
# Install required packages
!pip install -q pandas openpyxl scikit-learn anthropic


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score, confusion_matrix, classification_report
import json
import re
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded successfully")

## 2️⃣ Load Knowledge Components

In [ ]:
# Load knowledge components JSON
KNOWLEDGE_COMPONENTS_PATH = "/content/drive/MyDrive/climet-exam/knowledge_components.json"

with open(KNOWLEDGE_COMPONENTS_PATH, 'r', encoding='utf-8') as f:
    knowledge_components = json.load(f)

print("✅ Knowledge components loaded")
print(f"\nCategories:")
print(f"  Basic Level: {list(knowledge_components['temel_duzey']['kategoriler'].keys())}")
print(f"  Advanced Level: {list(knowledge_components['ileri_duzey']['kategoriler'].keys())}")

In [ ]:
# Extract term lists from knowledge components
def load_terms_from_knowledge_components(kc):
    """Extract term lists from knowledge components JSON"""

    basic_terms = []
    advanced_terms = []

    # Basic level terms
    for category, data in kc['temel_duzey']['kategoriler'].items():
        basic_terms.extend(data['terimler'])

    # Advanced level terms
    for category, data in kc['ileri_duzey']['kategoriler'].items():
        advanced_terms.extend(data['terimler'])

    # Lowercase and deduplicate
    basic_terms = list(set([term.lower() for term in basic_terms]))
    advanced_terms = list(set([term.lower() for term in advanced_terms]))

    return basic_terms, advanced_terms

BASIC_TERMS, ADVANCED_TERMS = load_terms_from_knowledge_components(knowledge_components)

print(f"\n✅ {len(BASIC_TERMS)} basic terms loaded")
print(f"✅ {len(ADVANCED_TERMS)} advanced terms loaded")
print(f"\nSample basic terms: {BASIC_TERMS[:5]}")
print(f"Sample advanced terms: {ADVANCED_TERMS[:5]}")

## 3️⃣ Few-shot Examples

In [ ]:
# Content scoring examples (Turkish)
CONTENT_EXAMPLES = [
    {
        "message": "naber",
        "score": 0,
        "reasoning": "Çevre/iklim ile ilgisi yok, sosyal selamlaşma. Konu dışı."
    },
    {
        "message": "Geri dönüşüm nedir?",
        "score": 1,
        "reasoning": "Sadece 'geri dönüşüm' (temel terim) var. İleri terim yok."
    },
    {
        "message": "Caretta carettalar nerede yaşar?",
        "score": 1,
        "reasoning": "'Caretta caretta' (temel biyolojik terim) ama başka kavram yok."
    },
    {
        "message": "Karbon ayak izi nedir?",
        "score": 2,
        "reasoning": "'Karbon ayak izi' ileri terimdir ama başka kavram yok."
    },
    {
        "message": "iklim kanununu uygulanabılmesı için nasıl bilinçlendirme işlemleri yapılabilir",
        "score": 2,
        "reasoning": "İleri terim var ama başka kavram yok."
    },
    {
        "message": "Yenilenebilir enerji kullanırsak karbon ayak izimiz azalır mı",
        "score": 3,
        "reasoning": "'Yenilenebilir enerji' ve 'karbon ayak izi' olmak üzere iki ayrı kavramı ilişkilendirmiş."
    },
    {
        "message": "Geri dönüştürülebilir enerjiler yerine kömür kullanırsak hava nasıl etkilenir",
        "score": 3,
        "reasoning": "'Kömür' (fosil yakıt/ileri), geri dönüştürülebilir enerjiler ile hava arasında bağlantı kurmuş."
    },
    {
        "message": "Fosil yakıt kullanımı sera gazı emisyonunu arttırırsa iklim değişikliğine yol açar mı",
        "score": 3,
        "reasoning": "Üç ileri terim kullanılmış."
    }
]

# Dialog scoring examples (Turkish)
DIALOG_EXAMPLES = [
    {
        "message": "tamam",
        "score": "A",
        "reasoning": "Tek kelime, diyalogu ilerletmiyor."
    },
    {
        "message": "anladım 👍",
        "score": "A",
        "reasoning": "Kısa onay + emoji, soru yok, akıl yürütme yok."
    },
    {
        "message": "Geri dönüşüm nedir?",
        "score": "B",
        "reasoning": "Basit tanımlayıcı soru, önceki mesaja atıf yok."
    },
    {
        "message": "Karbon ayak izi ne demek?",
        "score": "B",
        "reasoning": "Tanımlayıcı soru, yüzeysel bilgi arayışı."
    },
    {
        "message": "Plastik poşeti dönüştürmezsek ne yapabiliriz?",
        "score": "C",
        "reasoning": "Alternatif çözümler soruyor, diyalogu ilerletiyor ama akıl yürütme yok."
    },
    {
        "message": "Peki karbonayak izinin hayvanlar üzerine etkisi nasıl?",
        "score": "C",
        "reasoning": "'Peki' ile devam ediyor, daha fazla ayrıntı istiyor."
    },
    {
        "message": "Gelecekte enerji ihtiyacını karşılamak için neler yapılabilir?",
        "score": "C",
        "reasoning": "Bir amaca ulaşmaya yönelik öneri istiyor, diyalogu ilerletiyor ama akıl yürütme yok."
    },
    {
        "message": "Çevre kirliliği ne gibi sorunlara yol açabilir",
        "score": "C",
        "reasoning": "Gelecek ile ilgili nedensel soru soruyor, diyalogu ilerletiyor ama akıl yürütme yok."
    },
    {
        "message": "yenilenemez enerjiler tükenseydi ne olurdu",
        "score": "C",
        "reasoning": "Gelecek ile ilgili varsayımsal soru soruyor, diyalogu ilerletiyor ama akıl yürütme yok."
    },
    {
        "message": "bunun için kullanılabilecek kağıt ve karton malzemeler çabuk ıslanır, cam da hızlı kırılabilir. yani bu seçenekleri kullanmak mantıksız",
        "score": "D",
        "reasoning": "Neden-sonuç ilişkisi ve akıl yürütme var."
    },
    {
        "message": "Geri dönüşüm yaparsak doğal kaynakları koruruz çünkü ham madde kullanımı azalır",
        "score": "D",
        "reasoning": "'Çünkü' ile açık neden-sonuç akıl yürütmesi."
    }
]

print(f"✅ {len(CONTENT_EXAMPLES)} content examples loaded")
print(f"✅ {len(DIALOG_EXAMPLES)} dialog examples loaded")

## 4️⃣ Feature Extraction (v2.1.1 - Enhanced)

In [ ]:
def term_variants(term):
    """Generate possible variants of a term (Turkish morphology)"""
    variants = [term]

    # Handle multi-word terms
    if ' ' in term:
        words = term.split()
        variants.extend(words)

        # Generate morphological variants for each word
        for word in words:
            if len(word) > 4:
                # Common Turkish suffixes
                variants.extend([
                    word[:-1],      # Remove last char
                    word + 'lu',    # -lu suffix
                    word + 'lı',    # -lı suffix
                    word + 'sı',    # -sı suffix
                    word + 'si',    # -si suffix
                    word + 'nı',    # -nı suffix
                    word + 'ı',     # -ı suffix
                ])

    return variants

def extract_features(message: str) -> Dict:
    """Extract features from message - v2.1.1 Enhanced"""
    message_lower = message.lower()

    # Term detection with variants
    basic_terms_found = []
    for term in BASIC_TERMS:
        variants = term_variants(term)
        for variant in variants:
            if len(variant) > 3:  # Skip very short words
                # Match word with possible suffixes: \w* allows for inflections
                if re.search(r'\b' + re.escape(variant) + r'\w*\b', message_lower):
                    basic_terms_found.append(term)
                    break

    advanced_terms_found = []
    for term in ADVANCED_TERMS:
        variants = term_variants(term)
        for variant in variants:
            if len(variant) > 3:
                if re.search(r'\b' + re.escape(variant) + r'\w*\b', message_lower):
                    advanced_terms_found.append(term)
                    break

    # Special term mappings (fixes known issues)
    # "hidrolik" → "hidroelektrik"
    if 'hidrolik' in message_lower or 'su enerjisi' in message_lower:
        if 'hidroelektrik enerji' not in advanced_terms_found:
            advanced_terms_found.append('hidroelektrik enerji')

    # "tasarruflu" → "enerji tasarrufu"
    if 'tasarruflu' in message_lower and 'enerji' in message_lower:
        if 'enerji tasarrufu' not in advanced_terms_found:
            advanced_terms_found.append('enerji tasarrufu')

    # Remove duplicates
    basic_terms_found = list(set(basic_terms_found))
    advanced_terms_found = list(set(advanced_terms_found))

    # Reasoning indicators (Turkish)
    reasoning_indicators = [
        "çünkü", "bu nedenle", "dolayısıyla",
        "eğer", "ise", "o zaman", "olursa", "olmazsa",
        "sonuç olarak", "böylece", "nedeniyle", "sebebiyle",
        "bu sayede", "sayesinde", "yüzünden", "ötürü"
    ]
    has_reasoning = any(ind in message_lower for ind in reasoning_indicators)

    # Weak reasoning detection
    # "bu yüzden" is weak reasoning; if combined with strong action, it's not reasoning
    weak_reasoning = ["bu yüzden", "o yüzden"]
    strong_action = ["proje fikrim", "fikrim var", "planım", "önerim", "yapalım", "yapacağız"]

    has_weak_reasoning = any(wr in message_lower for wr in weak_reasoning)
    has_strong_action = any(sa in message_lower for sa in strong_action)

    # Weak reasoning + strong action = no reasoning (Dialog C not D)
    if has_weak_reasoning and has_strong_action:
        has_reasoning = False

    # Connection indicators (Turkish)
    connection_indicators = [
        "etkilenir", "etkiler", "etki", "ilişki", "bağlantı",
        "azalır", "azaltır", "artar", "artırır",
        "sebep olur", "neden olur", "sonucu", "sonucunda",
        "yerine", "karşılık", "ile birlikte", "beraber",
        "faydaları", "faydası", "yararları", "zararları"
    ]
    has_connection = any(conn in message_lower for conn in connection_indicators)

    # Question detection (Turkish)
    question_words = ["nedir", "ne demek", "nasıl", "neden", "niçin", "ne", "kim", "nerede", "ne zaman", "hangi"]
    is_question = message.strip().endswith("?") or any(qw in message_lower for qw in question_words)

    # Minimal message detection
    minimal_indicators = ["kısaca", "kısa", "özet", "özetçe"]
    is_very_short = any(mi in message_lower for mi in minimal_indicators) or len(message.split()) <= 2

    # Reference indicators (Turkish)
    reference_indicators = [
        "dediğin", "söylediğin", "bahsettiğin", "anlattığın",
        "peki", "ya", "başka", "ayrıca", "de ki", "sen dediğim",
        "daha önce", "az önce", "yukarıda", "yaptığın üzerine"
    ]
    has_reference = any(ref in message_lower for ref in reference_indicators)

    causal_and_effectual_indicators = [
        "yapmazsak", "nelere yol açar", "gelecekte", "etkileri",
        "nedenleri", "sebepleri"
    ]
    has_causal_and_effectual = any(cef in message_lower for cef in causal_and_effectual_indicators)

    # Action indicators (Turkish)
    action_indicators = [
        "yapabiliriz", "yapalım", "oluşturalım", "kuralım", "önerim",
        "proje", "plan", "adımlar", "önce", "sonra", "fikrim",
        "düşünüyorum", "yapacağız", "yapmalıyız", "öneriyorum", "fikrim var"
    ]
    has_action = any(act in message_lower for act in action_indicators)

    word_count = len(message.split())

    minimal_responses = ["evet", "hayır", "tamam", "ok", "olur", "anladım", "peki", "iyi"]
    is_minimal = message_lower.strip() in minimal_responses or is_very_short

    return {
        "basic_terms": basic_terms_found,
        "basic_term_count": len(basic_terms_found),
        "advanced_terms": advanced_terms_found,
        "advanced_term_count": len(advanced_terms_found),
        "has_reasoning": has_reasoning,
        "has_connection": has_connection,
        "is_question": is_question,
        "has_reference": has_reference,
        "has_causal_and_effectual": has_causal_and_effectual,
        "has_action": has_action,
        "word_count": word_count,
        "is_minimal": is_minimal
    }

# Test the feature extraction
test_msg = "Yenilenebilir enerji kullanırsak karbon ayak izimiz azalır"
features = extract_features(test_msg)
print(f"Test message: {test_msg}")
print(f"Advanced terms found: {features['advanced_terms'][:3]}")
print(f"Has reasoning: {features['has_reasoning']}")
print(f"Has connection: {features['has_connection']}")
print(f"Has cause and effect: {features['has_causal_and_effectual']}")

## 5️⃣ Rule-based Scoring (v2.1.1)

In [ ]:
def rule_based_content_score(features: Dict) -> int:
    """Rule-based content scoring"""
    basic_count = features["basic_term_count"]
    advanced_count = features["advanced_term_count"]
    has_reasoning = features["has_reasoning"]
    has_connection = features["has_connection"]

    total_terms = basic_count + advanced_count

    # Level 0: Off-topic
    if total_terms == 0:
        return 0

    # Level 3: Systemic connection
    # At least 1 advanced term + 2+ concepts + (reasoning OR connection)
    if advanced_count >= 1 and total_terms >= 2 and (has_reasoning or has_connection):
        return 3

    # Level 2: Advanced term present but no connection
    if advanced_count >= 1:
        return 2

    # Level 1: Only basic terms
    return 1

def rule_based_dialog_score(features: Dict) -> str:
    """Rule-based dialog scoring - v2.1.1 Enhanced"""

    # A: Minimal
    if features["is_minimal"]:
        return "A"

    # D: Strong reasoning OR (reasoning + long detailed plan)
    if features["has_reasoning"] and not features["has_action"]:
        return "D"
    if features["has_reasoning"] and features["word_count"] > 20:
        return "D"

    # C: Reference OR action (but no strong reasoning)
    if features["has_reference"] or features["has_action"]:
        return "C"

    # B: Default (simple questions)
    return "B"

# Test
print(f"Rule-based content score: {rule_based_content_score(features)}")
print(f"Rule-based dialog score: {rule_based_dialog_score(features)}")

## 6️⃣ RAG and Prompt Generation

In [ ]:
def find_similar_examples(message: str, examples: List[Dict], top_k: int = 3) -> List[Dict]:
    """Find most similar examples (word overlap)"""
    message_words = set(message.lower().split())

    similarities = []
    for ex in examples:
        ex_words = set(ex["message"].lower().split())
        overlap = len(message_words & ex_words)
        similarities.append((overlap, ex))

    similarities.sort(key=lambda x: x[0], reverse=True)
    return [ex for _, ex in similarities[:top_k]]

def create_content_prompt_turkish(message: str, features: Dict, examples: List[Dict]) -> str:
    """Create Turkish prompt for content evaluation"""

    examples_text = ""
    for i, ex in enumerate(examples, 1):
        examples_text += f"\n\nÖrnek {i}:\nMesaj: \"{ex['message']}\"\nPuan: {ex['score']}\nGerekçe: {ex['reasoning']}"

    # Highlight found terms
    basic_str = ', '.join(features['basic_terms'][:5]) if features['basic_terms'] else "YOK"
    advanced_str = ', '.join(features['advanced_terms'][:5]) if features['advanced_terms'] else "YOK"

    terms_info = f"""\nMesajda Bulunan Terimler:
- Temel terimler ({features['basic_term_count']}): {basic_str}
- İleri terimler ({features['advanced_term_count']}): {advanced_str}
- Akıl yürütme belirteci: {"VAR" if features['has_reasoning'] else "YOK"}
- İlişki belirteci: {"VAR" if features['has_connection'] else "YOK"}"""

    prompt = f"""Sen bir eğitim değerlendirme uzmanısın. Ortaokul öğrencilerinin iklim ve çevre konulu chatbot ile yaptığı konuşmaları değerlendiriyorsun.

İçerik Gelişmişliği Kriteri (0-3):

0 = Konu Dışı: Çevre, iklim, enerji, su veya tarım ile ilgili hiçbir terim yok.

1 = Temel Kavram: Sadece temel terimler var (örn: "su", "geri dönüşüm", "plastik").
    İleri terim YOK.

2 = İleri Kavram: En az 1 ileri terim var (örn: "karbon ayak izi", "iklim değişikliği",
    "sürdürülebilirlik", "enerji tasarrufu", "hidroelektrik enerji") AMA başka kavramla
    mantıksal bağlantı yok.

3 = Çoklu Kavram:
    ✓ En az 1 ileri terim VAR
    ✓ En az 2 farklı kavram VAR

{examples_text}

{terms_info}

ŞİMDİ DEĞERLENDİRİLECEK MESAJ:
\"{message}\"

ÖNEMLİ KURALLAR:
- Seviye 3 için MUTLAKA: (1) En az 1 ileri terim + (2) En az 2 kavram
- Birden fazla terim yoksa maksimum Seviye 2
- İleri terim yoksa maksimum Seviye 1
- Hiçbir terim yoksa Seviye 0
- "Enerji tasarrufu", "hidroelektrik", "iklim kanunu" gibi terimler İLERİ terimdir

CEVAP FORMATI (sadece JSON, başka hiçbir şey yazma):
{{\"score\": <0, 1, 2 veya 3>, \"reasoning\": \"<Türkçe gerekçe>\"}}"""

    return prompt

def create_dialog_prompt_turkish(message: str, features: Dict, examples: List[Dict]) -> str:
    """Create Turkish prompt for dialog evaluation"""

    examples_text = ""
    for i, ex in enumerate(examples, 1):
        examples_text += f"\n\nÖrnek {i}:\nMesaj: \"{ex['message']}\"\nPuan: {ex['score']}\nGerekçe: {ex['reasoning']}"

    features_info = f"""\nMesaj Özellikleri:
- Kelime sayısı: {features['word_count']}
- Minimal mesaj: {"EVET" if features['is_minimal'] else "HAYIR"}
- Soru mu: {"EVET" if features['is_question'] else "HAYIR"}
- Akıl yürütme var: {"EVET" if features['has_reasoning'] else "HAYIR"}
- Referans var: {"EVET" if features['has_reference'] else "HAYIR"}
- Etkisel soru var: {"EVET" if features['has_causal_and_effectual'] else "HAYIR"}
- Eylem/öneri var: {"EVET" if features['has_action'] else "HAYIR"}"""

    prompt = f"""Sen bir eğitim değerlendirme uzmanısın. Ortaokul öğrencilerinin iklim ve çevre konulu chatbot ile yaptığı konuşmaları değerlendiriyorsun.

Diyalog Seviyesi Kriteri (A-D):

A = Minimal / İlerlemeyen:
    - Tek kelime ("evet", "tamam"), emoji, diyalogu ilerletmeyen mesaj
    - "Kısaca", "özetçe" gibi kısa talep kelimeleri
    - 2 kelime veya daha az

B = Basit Soru-Cevap:
    - "X nedir?" tarzı tanımlayıcı sorular, yüzeysel sorular
    - Tek cümle cevap, önceki mesaja atıf yok
    - VEYA Süreç odaklı sorular (örneğin, “… nasıl gerçekleşir?”)

C = Diyalogu İlerletici:
    - Referans var ("peki", "dediğin gibi", "sen dediğim", "yaptığın üzerine")
    - VEYA öneri/fikir sunuyor ("proje", "plan", "yapabiliriz", "fikrim var")
    - VEYA Etkisel soru var
    - VEYA Sebep-sonuç soruları (örneğin, “…'nın etkisi nedir?”)
    - VEYA Kavramlar veya olaylar arasındaki ilişkileri sorgulama
    - VEYA Temel nedenleri veya sonuçları sorgulama
    - AMA güçlü akıl yürütme yok
    - NOT: "Bu yüzden" + proje fikri = C (zayıf akıl yürütme + güçlü eylem)

D = Akıl Yürütme / Plan:
    - "Çünkü", "eğer...ise", "bu nedenle", "dolayısıyla" gibi GÜÇLÜ akıl yürütme yapıları VAR
    - VEYA detaylı, adım adım eylem planı VAR (20+ kelime)
    - NOT: Sadece "bu yüzden" zayıf akıl yürütmedir, eğer proje fikri varsa C

{examples_text}

{features_info}

ŞİMDİ DEĞERLENDİRİLECEK MESAJ:
\"{message}\"

ÖNEMLİ KURALLAR:
- D için MUTLAKA: Güçlü akıl yürütme ("çünkü", "eğer", "bu nedenle") VEYA detaylı plan
- "Bu yüzden" + proje fikri = C (zayıf akıl yürütme)
- "Kısaca" veya tek kelime = A (minimal)
- C için: Referans VEYA öneri var ama güçlü akıl yürütme yok
- B için: Basit soru veya kısa cevap

CEVAP FORMATI (sadece JSON, başka hiçbir şey yazma):
{{\"score\": \"<A, B, C veya D harfi>\", \"reasoning\": \"<Türkçe gerekçe>\"}}"""

    return prompt

print("✅ Prompt functions ready")

## 7️⃣ API Setup

In [ ]:
# Choose API
USE_API = "anthropic"

if USE_API == "anthropic":
    from google.colab import userdata
    import os

    try:
        ANTHROPIC_API_KEY = userdata.get("anthropic")
    except:
        ANTHROPIC_API_KEY = input("Enter Anthropic API Key: ")

    os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
    print("✅ Anthropic API ready")

## 8️⃣ LLM Scoring

In [ ]:
def score_with_llm(message: str, score_type: str = "content", use_api: str = "anthropic") -> Dict:
    """Score message with LLM"""

    features = extract_features(message)

    if score_type == "content":
        similar_examples = find_similar_examples(message, CONTENT_EXAMPLES, top_k=3)
        prompt = create_content_prompt_turkish(message, features, similar_examples)
        rule_score = rule_based_content_score(features)
    else:
        similar_examples = find_similar_examples(message, DIALOG_EXAMPLES, top_k=3)
        prompt = create_dialog_prompt_turkish(message, features, similar_examples)
        rule_score = rule_based_dialog_score(features)

    try:
        if use_api == "anthropic":
            import anthropic
            client = anthropic.Anthropic()
            response = client.messages.create(
                model="claude-sonnet-4-20250514",
                max_tokens=512,
                messages=[{"role": "user", "content": prompt}]
            )
            response_text = response.content[0].text.strip()

        # Parse JSON
        response_text = re.sub(r'```json\s*|\s*```', '', response_text).strip()
        result = json.loads(response_text)

        # Validate
        if score_type == "content":
            if result["score"] not in [0, 1, 2, 3]:
                result["score"] = rule_score
                result["reasoning"] = f"Invalid score, using rule-based: {rule_score}"
        else:
            if result["score"] not in ["A", "B", "C", "D"]:
                result["score"] = rule_score
                result["reasoning"] = f"Invalid score, using rule-based: {rule_score}"

        result["method"] = "llm"
        return result

    except Exception as e:
        print(f"  ⚠ LLM error: {str(e)[:50]}...")
        return {
            "score": rule_score,
            "reasoning": "Rule-based evaluation (LLM error)",
            "method": "rule_based"
        }

# Test
result = score_with_llm(test_msg, "content", USE_API)
print(f"Test result: {result}")

## 9️⃣ Dataset Processing

In [ ]:
# Load dataset
EXCEL_PATH = "/content/drive/MyDrive/climet-exam/example-genisletilmis-revised.xlsx"

df = pd.read_excel(EXCEL_PATH)
print(f"✅ {len(df)} messages loaded")
print(f"\nContent distribution: {dict(df['content'].value_counts().sort_index())}")
print(f"Dialog distribution: {dict(df['dialog'].value_counts().sort_index())}")

In [ ]:
def process_dataset(df: pd.DataFrame, use_api: str = "anthropic", sample_size: int = None) -> pd.DataFrame:
    """Process and score dataset"""

    if sample_size:
        df = df.sample(min(sample_size, len(df)), random_state=42).copy()
    else:
        df = df.copy()

    df['content_llm'] = None
    df['content_reasoning'] = None
    df['content_method'] = None
    df['dialog_llm'] = None
    df['dialog_reasoning'] = None
    df['dialog_method'] = None

    print(f"\n🔄 Processing {len(df)} messages...\n")

    for idx, row in df.iterrows():
        message = str(row['message'])
        print(f"[{idx+1}/{len(df)}] {message[:60]}...")

        # Content scoring
        content_result = score_with_llm(message, "content", use_api)
        df.at[idx, 'content_llm'] = content_result['score']
        df.at[idx, 'content_reasoning'] = content_result['reasoning']
        df.at[idx, 'content_method'] = content_result['method']
        print(f"  Content: {row['content']} → {content_result['score']} ({content_result['method']})")

        # Dialog scoring
        dialog_result = score_with_llm(message, "dialog", use_api)
        df.at[idx, 'dialog_llm'] = dialog_result['score']
        df.at[idx, 'dialog_reasoning'] = dialog_result['reasoning']
        df.at[idx, 'dialog_method'] = dialog_result['method']
        print(f"  Dialog:  {row['dialog']} → {dialog_result['score']} ({dialog_result['method']})\n")

    return df

print("✅ Processing function ready")

In [ ]:
# Process dataset (start with sample for testing)
SAMPLE_SIZE = 20

df_scored = process_dataset(df, USE_API, SAMPLE_SIZE)
print("\n✅ Processing complete!")

## 🔟 Cohen's Kappa Evaluation (Error-safe)

In [ ]:
def clean_labels(human_labels, llm_labels, label_type="content"):
    """Clean invalid labels to prevent Cohen's Kappa errors"""

    valid_indices = []

    for i in range(len(human_labels)):
        h_label = human_labels.iloc[i] if hasattr(human_labels, 'iloc') else human_labels[i]
        l_label = llm_labels.iloc[i] if hasattr(llm_labels, 'iloc') else llm_labels[i]

        # Check for None/NaN
        if pd.isna(h_label) or pd.isna(l_label) or h_label is None or l_label is None:
            continue

        # Validate content scores (0-3)
        if label_type == "content":
            try:
                h_int = int(h_label)
                l_int = int(l_label)
                if h_int in [0, 1, 2, 3] and l_int in [0, 1, 2, 3]:
                    valid_indices.append(i)
            except (ValueError, TypeError):
                continue

        # Validate dialog scores (A-D)
        else:
            h_str = str(h_label).strip().upper()
            l_str = str(l_label).strip().upper()
            if h_str in ['A', 'B', 'C', 'D'] and l_str in ['A', 'B', 'C', 'D']:
                valid_indices.append(i)

    return valid_indices

def evaluate_agreement(df: pd.DataFrame) -> Dict:
    """Calculate Cohen's Kappa (error-safe)"""

    results = {}

    # Content evaluation
    print("\n" + "="*80)
    print("CONTENT LEVEL EVALUATION")
    print("="*80)

    valid_content_indices = clean_labels(df['content'], df['content_llm'], "content")

    if len(valid_content_indices) == 0:
        print("\n⚠️  No valid content scores found!")
        results['content'] = None
    else:
        content_human = df.iloc[valid_content_indices]['content'].values.astype(int)
        content_llm = df.iloc[valid_content_indices]['content_llm'].values.astype(int)

        content_kappa = cohen_kappa_score(content_human, content_llm)
        content_accuracy = np.mean(content_human == content_llm)
        content_conf_matrix = confusion_matrix(content_human, content_llm, labels=[0,1,2,3])

        print(f"\nCohen's Kappa: {content_kappa:.4f}")
        print(f"Accuracy: {content_accuracy:.2%}")
        print(f"Valid samples: {len(valid_content_indices)} / {len(df)}")
        print(f"\nConfusion Matrix:")
        print("         LLM_0  LLM_1  LLM_2  LLM_3")
        for i in range(4):
            print(f"Human_{i}:  {content_conf_matrix[i,0]:4d}  {content_conf_matrix[i,1]:4d}  "
                  f"{content_conf_matrix[i,2]:4d}  {content_conf_matrix[i,3]:4d}")

        print(f"\nDetailed Report:")
        print(classification_report(content_human, content_llm, labels=[0,1,2,3],
                                    target_names=['0-OffTopic', '1-Basic', '2-Advanced', '3-Systemic'],
                                    zero_division=0))

        results['content'] = {
            'kappa': content_kappa,
            'accuracy': content_accuracy,
            'n_valid': len(valid_content_indices)
        }

    # Dialog evaluation
    print("\n" + "="*80)
    print("DIALOG LEVEL EVALUATION")
    print("="*80)

    valid_dialog_indices = clean_labels(df['dialog'], df['dialog_llm'], "dialog")

    if len(valid_dialog_indices) == 0:
        print("\n⚠️  No valid dialog scores found!")
        results['dialog'] = None
    else:
        dialog_human = df.iloc[valid_dialog_indices]['dialog'].values
        dialog_llm = df.iloc[valid_dialog_indices]['dialog_llm'].values

        # Convert letters to numbers
        dialog_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
        dialog_human_num = np.array([dialog_map[str(x).strip().upper()] for x in dialog_human])
        dialog_llm_num = np.array([dialog_map[str(x).strip().upper()] for x in dialog_llm])

        dialog_kappa = cohen_kappa_score(dialog_human_num, dialog_llm_num)
        dialog_accuracy = np.mean(dialog_human_num == dialog_llm_num)
        dialog_conf_matrix = confusion_matrix(dialog_human_num, dialog_llm_num, labels=[0,1,2,3])

        print(f"\nCohen's Kappa: {dialog_kappa:.4f}")
        print(f"Accuracy: {dialog_accuracy:.2%}")
        print(f"Valid samples: {len(valid_dialog_indices)} / {len(df)}")
        print(f"\nConfusion Matrix:")
        print("       LLM_A  LLM_B  LLM_C  LLM_D")
        for i, label in enumerate(['A', 'B', 'C', 'D']):
            print(f"Human_{label}:  {dialog_conf_matrix[i,0]:4d}  {dialog_conf_matrix[i,1]:4d}  "
                  f"{dialog_conf_matrix[i,2]:4d}  {dialog_conf_matrix[i,3]:4d}")

        print(f"\nDetailed Report:")
        print(classification_report(dialog_human_num, dialog_llm_num, labels=[0,1,2,3],
                                    target_names=['A-Minimal', 'B-Simple', 'C-Advancing', 'D-Reasoning'],
                                    zero_division=0))

        results['dialog'] = {
            'kappa': dialog_kappa,
            'accuracy': dialog_accuracy,
            'n_valid': len(valid_dialog_indices)
        }

    # Summary
    print("\n" + "="*80)
    print("SUMMARY")
    print("="*80)

    if results['content']:
        print(f"Content Kappa: {results['content']['kappa']:.4f} - Accuracy: {results['content']['accuracy']:.2%}")
    if results['dialog']:
        print(f"Dialog Kappa:  {results['dialog']['kappa']:.4f} - Accuracy: {results['dialog']['accuracy']:.2%}")

    if results['content'] and results['dialog']:
        avg_kappa = (results['content']['kappa'] + results['dialog']['kappa']) / 2
        print(f"\n🎯 Average Kappa: {avg_kappa:.4f}")

    return results

results = evaluate_agreement(df_scored)

## 1️⃣1️⃣ Error Analysis

In [ ]:
def analyze_errors(df: pd.DataFrame, top_n: int = 10):
    """Analyze prediction errors"""

    df['content_error'] = df['content'] != df['content_llm']
    df['dialog_error'] = df['dialog'].str.strip().str.upper() != df['dialog_llm'].str.strip().str.upper()

    content_errors = df[df['content_error']]
    dialog_errors = df[df['dialog_error']]

    print("\n" + "="*80)
    print("ERROR ANALYSIS")
    print("="*80)

    print(f"\nContent errors: {len(content_errors)} / {len(df)}")
    print(f"Dialog errors: {len(dialog_errors)} / {len(df)}")

    if len(content_errors) > 0:
        print(f"\n📊 Content Error Examples:")
        for i, (idx, row) in enumerate(content_errors.head(top_n).iterrows(), 1):
            print(f"\n{i}. {row['message'][:80]}")
            print(f"   Human: {row['content']} | LLM: {row['content_llm']}")
            print(f"   {row['content_reasoning'][:80]}")

    if len(dialog_errors) > 0:
        print(f"\n\n📊 Dialog Error Examples:")
        for i, (idx, row) in enumerate(dialog_errors.head(top_n).iterrows(), 1):
            print(f"\n{i}. {row['message'][:80]}")
            print(f"   Human: {row['dialog']} | LLM: {row['dialog_llm']}")
            print(f"   {row['dialog_reasoning'][:80]}")

analyze_errors(df_scored, 5)

## 1️⃣2️⃣ Evaluate and Label Student Dialogs

In [ ]:
# ============================================================================
# BATCH PROCESSING FOR STUDENT DIALOGS
# ============================================================================
# Add this cell to your existing v2.1.1 notebook
# Does NOT modify any existing functions - uses them as-is
# ============================================================================

import os
from pathlib import Path
from tqdm.auto import tqdm
import time

# ============================================================================
# CONFIGURATION
# ============================================================================

# Paths (UPDATE THESE)
BASE_PATH = Path("/content/drive/MyDrive/climet-exam/student_dialogs")
OUTPUT_BASE = BASE_PATH / "labeled"

# Processing settings
BATCH_SIZE = 50  # Auto-save checkpoint every N messages
SKIP_EXISTING = True  # Skip files that are already processed

print("="*80)
print("🌍 BATCH PROCESSING - STUDENT DIALOGS")
print("="*80)
print(f"\nInput folder:  {BASE_PATH}")
print(f"Output folder: {OUTPUT_BASE}")
print(f"Batch size:    {BATCH_SIZE} messages")
print(f"Skip existing: {SKIP_EXISTING}")

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def process_single_csv(csv_path: Path, output_path: Path, use_api: str = "anthropic"):
    """
    Process a single CSV file
    - Filters only student messages (Sender == 'student')
    - Leaves bot rows empty in labeled columns
    - Preserves all original columns
    """

    # Load CSV
    try:
        df = pd.read_csv(csv_path, encoding='utf-8-sig')
    except:
        try:
            df = pd.read_csv(csv_path, encoding='latin-1')
        except Exception as e:
            print(f"   ❌ Cannot read file: {e}")
            return None

    print(f"\n📄 {csv_path.name}")
    print(f"   Total rows: {len(df)}")
    print(f"   Columns: {', '.join(df.columns[:5])}{'...' if len(df.columns) > 5 else ''}")

    # Check for required columns
    if 'Sender' not in df.columns:
        print(f"   ❌ 'Sender' column not found!")
        return None

    if 'Message' not in df.columns:
        print(f"   ❌ 'Message' column not found!")
        return None

    # Count student messages
    n_students = len(df[df['Sender'] == 'student'])
    print(f"   Student messages: {n_students} / {len(df)}")

    if n_students == 0:
        print(f"   ⚠️  No student messages found, skipping...")
        return None

    # Add empty label columns for ALL rows
    df['content_label'] = None
    df['dialog_label'] = None
    df['knowledge_components'] = None
    df['content_reasoning'] = None
    df['dialog_reasoning'] = None
    df['scoring_method'] = None

    # Statistics
    stats = {
        'total': n_students,
        'processed': 0,
        'llm_used': 0,
        'rule_based': 0,
        'errors': 0
    }

    # Process ONLY student messages
    student_indices = df[df['Sender'] == 'student'].index

    for idx in tqdm(student_indices, desc="  Processing", leave=False):
        row = df.loc[idx]
        message = str(row['Message'])

        # Skip empty messages
        if not message or message == 'nan' or len(message.strip()) == 0:
            stats['errors'] += 1
            continue

        try:
            # Extract features (uses all existing functions)
            features = extract_features(message)

            # Store knowledge components as JSON
            kc_list = list(features.get('advanced_terms', []))[:10]  # Top 10 KCs
            df.at[idx, 'knowledge_components'] = json.dumps(kc_list, ensure_ascii=False)

            # Score content (uses existing score_with_llm function)
            content_result = score_with_llm(message, "content", use_api)
            df.at[idx, 'content_label'] = content_result['score']
            df.at[idx, 'content_reasoning'] = content_result['reasoning']

            # Score dialog (uses existing score_with_llm function)
            dialog_result = score_with_llm(message, "dialog", use_api)
            df.at[idx, 'dialog_label'] = dialog_result['score']
            df.at[idx, 'dialog_reasoning'] = dialog_result['reasoning']

            # Track method
            method = f"{content_result['method']}/{dialog_result['method']}"
            df.at[idx, 'scoring_method'] = method

            # Update stats
            stats['processed'] += 1
            if 'llm' in method:
                stats['llm_used'] += 1
            else:
                stats['rule_based'] += 1

            # Auto-save checkpoint
            if stats['processed'] % BATCH_SIZE == 0:
                df.to_csv(output_path, index=False, encoding='utf-8-sig')
                print(f"   💾 Checkpoint: {stats['processed']}/{stats['total']}")

        except Exception as e:
            print(f"   ⚠️  Error at row {idx}: {str(e)[:50]}")
            stats['errors'] += 1
            continue

    # Final save
    df.to_csv(output_path, index=False, encoding='utf-8-sig')

    print(f"   ✅ Complete!")
    print(f"   Processed: {stats['processed']}/{stats['total']}")
    print(f"   LLM: {stats['llm_used']}, Rule-based: {stats['rule_based']}, Errors: {stats['errors']}")

    return stats

def process_all_folders():
    """
    Process all subfolders in BASE_PATH
    Creates parallel structure in OUTPUT_BASE
    """

    # Find all subfolders
    subfolders = [f for f in BASE_PATH.iterdir()
                  if f.is_dir() and not f.name.startswith('.') and f.name != 'labeled']

    if not subfolders:
        print("\n❌ No subfolders found!")
        return

    print(f"\n🔍 Found {len(subfolders)} subfolders:")
    for folder in subfolders:
        csv_count = len(list(folder.glob("*.csv")))
        print(f"   📁 {folder.name} ({csv_count} CSV files)")

    # Overall statistics
    overall_stats = {
        'folders': 0,
        'files_processed': 0,
        'files_skipped': 0,
        'total_messages': 0,
        'total_llm': 0,
        'total_rule_based': 0,
        'total_errors': 0
    }

    start_time = time.time()

    # Process each subfolder
    for subfolder in subfolders:
        print(f"\n{'='*80}")
        print(f"📁 FOLDER: {subfolder.name}")
        print(f"{'='*80}")

        # Create output folder
        output_folder = OUTPUT_BASE / subfolder.name
        output_folder.mkdir(parents=True, exist_ok=True)

        # Find all CSV files
        csv_files = list(subfolder.glob("*.csv"))
        print(f"   Found {len(csv_files)} CSV files")

        # Process each CSV
        for csv_file in csv_files:
            output_file = output_folder / csv_file.name

            # Skip if already processed
            if SKIP_EXISTING and output_file.exists():
                print(f"\n   ⏭️  {csv_file.name} (already exists)")
                overall_stats['files_skipped'] += 1
                continue

            # Process file
            try:
                stats = process_single_csv(csv_file, output_file, USE_API)

                if stats:
                    overall_stats['files_processed'] += 1
                    overall_stats['total_messages'] += stats['total']
                    overall_stats['total_llm'] += stats['llm_used']
                    overall_stats['total_rule_based'] += stats['rule_based']
                    overall_stats['total_errors'] += stats['errors']

            except Exception as e:
                print(f"   ❌ Error: {str(e)[:100]}")
                continue

        overall_stats['folders'] += 1

    # Final summary
    elapsed = time.time() - start_time

    print(f"\n{'='*80}")
    print("📊 BATCH PROCESSING COMPLETE")
    print(f"{'='*80}")
    print(f"\nFolders processed:     {overall_stats['folders']}/{len(subfolders)}")
    print(f"Files processed:       {overall_stats['files_processed']}")
    print(f"Files skipped:         {overall_stats['files_skipped']}")
    print(f"Total student messages: {overall_stats['total_messages']}")
    print(f"LLM scoring:           {overall_stats['total_llm']} ({overall_stats['total_llm']/max(overall_stats['total_messages'],1)*100:.1f}%)")
    print(f"Rule-based scoring:    {overall_stats['total_rule_based']} ({overall_stats['total_rule_based']/max(overall_stats['total_messages'],1)*100:.1f}%)")
    print(f"Errors:                {overall_stats['total_errors']}")
    print(f"\nTime elapsed:          {elapsed/60:.1f} minutes")
    print(f"Avg time per message:  {elapsed/max(overall_stats['total_messages'],1):.2f} seconds")

    print(f"\n✅ All labeled files saved to: {OUTPUT_BASE}")

    return overall_stats

# ============================================================================
# RUN BATCH PROCESSING
# ============================================================================

print("\n" + "="*80)
print("🚀 STARTING BATCH PROCESSING")
print("="*80)
print("\nThis will:")
print("  1. Find all CSV files in subfolders")
print("  2. Process ONLY rows where Sender == 'student'")
print("  3. Leave bot message rows empty in label columns")
print("  4. Save labeled CSVs with all original columns")
print("  5. Auto-save checkpoints every 50 messages")
print("\nPress Ctrl+C to cancel or wait 5 seconds to start...")

# Safety delay
time.sleep(5)

# Create output base directory
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

# Run processing
try:
    results = process_all_folders()
    print("\n✅ ALL PROCESSING COMPLETE!")
except KeyboardInterrupt:
    print("\n\n⚠️  Processing cancelled by user")
except Exception as e:
    print(f"\n\n❌ Fatal error: {e}")

print("\n" + "="*80)

# ============================================================================
# VERIFICATION
# ============================================================================

print("\n🔍 Quick verification of labeled files...")

labeled_files = list(OUTPUT_BASE.rglob("*.csv"))
if labeled_files:
    print(f"\n✅ Found {len(labeled_files)} labeled CSV files")

    # Show sample from first file
    sample_file = labeled_files[0]
    df_sample = pd.read_csv(sample_file, nrows=10)

    print(f"\n📋 Sample from: {sample_file.name}")
    print("\nColumns:", list(df_sample.columns))

    # Show a labeled student message
    student_rows = df_sample[df_sample['Sender'] == 'student']
    if len(student_rows) > 0:
        row = student_rows.iloc[0]
        print(f"\n✓ Student message example:")
        print(f"  Message: {row['Message'][:60]}...")
        print(f"  Content: {row['content_label']}")
        print(f"  Dialog: {row['dialog_label']}")
        print(f"  KCs: {row['knowledge_components'][:60]}...")

    # Show a bot message (should be empty)
    bot_rows = df_sample[df_sample['Sender'] != 'student']
    if len(bot_rows) > 0:
        row = bot_rows.iloc[0]
        print(f"\n✓ Bot message example (labels should be empty):")
        print(f"  Sender: {row['Sender']}")
        print(f"  Content label: {row['content_label']}")
        print(f"  Dialog label: {row['dialog_label']}")
else:
    print("\n⚠️  No labeled files found!")

print("\n" + "="*80)
print("✅ BATCH PROCESSING SCRIPT COMPLETE")
print("="*80)

## 🧪 Bugfix Verification Tests


In [ ]:
print("="*80)
print("BUGFIX VERIFICATION TESTS")
print("="*80)

# Test 1: Energy efficiency detection
print("\n🧪 Test 1: Energy efficiency term detection")
test1 = "Enerjiyi tasarruflu kullanmak neden önemlidir?"
f1 = extract_features(test1)
print(f"Message: {test1}")
print(f"Advanced terms: {f1['advanced_terms']}")
print(f"Expected: ['enerji tasarrufu']")
print(f"✓ PASS" if 'enerji tasarrufu' in f1['advanced_terms'] else "✗ FAIL")

# Test 2: Hydroelectric mapping
print("\n🧪 Test 2: Hidrolik → hidroelektrik mapping")
test2 = "Su (hidrolik) enerjisi hangi kaynaklardan elde edilir?"
f2 = extract_features(test2)
print(f"Message: {test2}")
print(f"Advanced terms: {f2['advanced_terms']}")
print(f"Expected: ['hidroelektrik enerji']")
print(f"✓ PASS" if 'hidroelektrik enerji' in f2['advanced_terms'] else "✗ FAIL")

# Test 3: Weak reasoning + strong action = Dialog C
print("\n🧪 Test 3: Weak reasoning + strong action = C (not D)")
test3 = "çöpler doğayı kirletiyor bu yüzden bir proje fikrim var"
f3 = extract_features(test3)
score3 = rule_based_dialog_score(f3)
print(f"Message: {test3}")
print(f"Has reasoning: {f3['has_reasoning']}")
print(f"Has action: {f3['has_action']}")
print(f"Dialog score: {score3}")
print(f"Expected: C")
print(f"✓ PASS" if score3 == "C" else f"✗ FAIL (got {score3})")

# Test 4: "kısaca" = Minimal (Dialog A)
print("\n🧪 Test 4: 'kısaca' detected as minimal (Dialog A)")
test4 = "sürdürülebilirlik kısaca"
f4 = extract_features(test4)
score4 = rule_based_dialog_score(f4)
print(f"Message: {test4}")
print(f"Is minimal: {f4['is_minimal']}")
print(f"Dialog score: {score4}")
print(f"Expected: A")
print(f"✓ PASS" if score4 == "A" else f"✗ FAIL (got {score4})")

# Test 5: Single word = Minimal (Dialog A)
print("\n🧪 Test 5: Single word = minimal (Dialog A)")
test5 = "sürdürülebilirlik"
f5 = extract_features(test5)
score5 = rule_based_dialog_score(f5)
print(f"Message: {test5}")
print(f"Is minimal: {f5['is_minimal']}")
print(f"Dialog score: {score5}")
print(f"Expected: A")
print(f"✓ PASS" if score5 == "A" else f"✗ FAIL (got {score5})")

print("\n" + "="*80)
print("All tests should PASS for v2.1.1 to work correctly")
print("="*80)

# **Kappa Scores**

In [ ]:
# Load dataset
EXCEL_PATH = "/content/drive/MyDrive/climet-exam/evaluations/sinif_j.5d_ogrenciler_2025-10-06-eval.xlsx"

df = pd.read_excel(EXCEL_PATH)
print(f"✅ {len(df)} messages loaded")
print(f"\nContent distribution: {dict(df['content'].value_counts().sort_index())}")
print(f"Dialog distribution: {dict(df['dialog'].value_counts().sort_index())}")

In [ ]:
def clean_labels(human_labels, llm_labels, label_type="content"):
    """Clean invalid labels to prevent Cohen's Kappa errors"""

    valid_indices = []

    for i in range(len(human_labels)):
        h_label = human_labels.iloc[i] if hasattr(human_labels, 'iloc') else human_labels[i]
        l_label = llm_labels.iloc[i] if hasattr(llm_labels, 'iloc') else llm_labels[i]

        # Check for None/NaN
        if pd.isna(h_label) or pd.isna(l_label) or h_label is None or l_label is None:
            continue

        # Validate content scores (0-3)
        if label_type == "content":
            try:
                h_int = int(h_label)
                l_int = int(l_label)
                if h_int in [0, 1, 2, 3] and l_int in [0, 1, 2, 3]:
                    valid_indices.append(i)
            except (ValueError, TypeError):
                continue

        # Validate dialog scores (A-D)
        else:
            h_str = str(h_label).strip().upper()
            l_str = str(l_label).strip().upper()
            if h_str in ['A', 'B', 'C', 'D'] and l_str in ['A', 'B', 'C', 'D']:
                valid_indices.append(i)

    return valid_indices

def evaluate_agreement(df: pd.DataFrame) -> Dict:
    """Calculate Cohen's Kappa (error-safe)"""

    results = {}

    # Content evaluation
    print("\n" + "="*80)
    print("CONTENT LEVEL EVALUATION")
    print("="*80)

    valid_content_indices = clean_labels(df['content'], df['content_llm'], "content")

    if len(valid_content_indices) == 0:
        print("\n⚠️  No valid content scores found!")
        results['content'] = None
    else:
        content_human = df.iloc[valid_content_indices]['content'].values.astype(int)
        content_llm = df.iloc[valid_content_indices]['content_llm'].values.astype(int)

        content_kappa = cohen_kappa_score(content_human, content_llm)
        content_accuracy = np.mean(content_human == content_llm)
        content_conf_matrix = confusion_matrix(content_human, content_llm, labels=[0,1,2,3])

        print(f"\nCohen's Kappa: {content_kappa:.4f}")
        print(f"Accuracy: {content_accuracy:.2%}")
        print(f"Valid samples: {len(valid_content_indices)} / {len(df)}")
        print(f"\nConfusion Matrix:")
        print("         LLM_0  LLM_1  LLM_2  LLM_3")
        for i in range(4):
            print(f"Human_{i}:  {content_conf_matrix[i,0]:4d}  {content_conf_matrix[i,1]:4d}  "
                  f"{content_conf_matrix[i,2]:4d}  {content_conf_matrix[i,3]:4d}")

        print(f"\nDetailed Report:")
        print(classification_report(content_human, content_llm, labels=[0,1,2,3],
                                    target_names=['0-OffTopic', '1-Basic', '2-Advanced', '3-Systemic'],
                                    zero_division=0))

        results['content'] = {
            'kappa': content_kappa,
            'accuracy': content_accuracy,
            'n_valid': len(valid_content_indices)
        }

    # Dialog evaluation
    print("\n" + "="*80)
    print("DIALOG LEVEL EVALUATION")
    print("="*80)

    valid_dialog_indices = clean_labels(df['dialog'], df['dialog_llm'], "dialog")

    if len(valid_dialog_indices) == 0:
        print("\n⚠️  No valid dialog scores found!")
        results['dialog'] = None
    else:
        dialog_human = df.iloc[valid_dialog_indices]['dialog'].values
        dialog_llm = df.iloc[valid_dialog_indices]['dialog_llm'].values

        # Convert letters to numbers
        dialog_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
        dialog_human_num = np.array([dialog_map[str(x).strip().upper()] for x in dialog_human])
        dialog_llm_num = np.array([dialog_map[str(x).strip().upper()] for x in dialog_llm])

        dialog_kappa = cohen_kappa_score(dialog_human_num, dialog_llm_num)
        dialog_accuracy = np.mean(dialog_human_num == dialog_llm_num)
        dialog_conf_matrix = confusion_matrix(dialog_human_num, dialog_llm_num, labels=[0,1,2,3])

        print(f"\nCohen's Kappa: {dialog_kappa:.4f}")
        print(f"Accuracy: {dialog_accuracy:.2%}")
        print(f"Valid samples: {len(valid_dialog_indices)} / {len(df)}")
        print(f"\nConfusion Matrix:")
        print("       LLM_A  LLM_B  LLM_C  LLM_D")
        for i, label in enumerate(['A', 'B', 'C', 'D']):
            print(f"Human_{label}:  {dialog_conf_matrix[i,0]:4d}  {dialog_conf_matrix[i,1]:4d}  "
                  f"{dialog_conf_matrix[i,2]:4d}  {dialog_conf_matrix[i,3]:4d}")

        print(f"\nDetailed Report:")
        print(classification_report(dialog_human_num, dialog_llm_num, labels=[0,1,2,3],
                                    target_names=['A-Minimal', 'B-Simple', 'C-Advancing', 'D-Reasoning'],
                                    zero_division=0))

        results['dialog'] = {
            'kappa': dialog_kappa,
            'accuracy': dialog_accuracy,
            'n_valid': len(valid_dialog_indices)
        }

    # Summary
    print("\n" + "="*80)
    print("SUMMARY")
    print("="*80)

    if results['content']:
        print(f"Content Kappa: {results['content']['kappa']:.4f} - Accuracy: {results['content']['accuracy']:.2%}")
    if results['dialog']:
        print(f"Dialog Kappa:  {results['dialog']['kappa']:.4f} - Accuracy: {results['dialog']['accuracy']:.2%}")

    if results['content'] and results['dialog']:
        avg_kappa = (results['content']['kappa'] + results['dialog']['kappa']) / 2
        print(f"\n🎯 Average Kappa: {avg_kappa:.4f}")

    return results

results = evaluate_agreement(df)

In [ ]:
from sklearn.metrics import cohen_kappa_score
import numpy as np
import pandas as pd

# ------------------ yardımcı ------------------
def compute_kappa_pair(df, human_col, llm_col, label_type):
    valid_idx = clean_labels(df[human_col], df[llm_col], label_type)
    if not valid_idx:
        return None

    if label_type == "content":
        h = df.loc[valid_idx, human_col].astype(int).values
        l = df.loc[valid_idx, llm_col].astype(int).values
        return cohen_kappa_score(h, l, weights="quadratic")  # önerilen
    else:
        dialog_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
        h = df.loc[valid_idx, human_col].astype(str).str.upper().map(dialog_map).values
        l = df.loc[valid_idx, llm_col].astype(str).str.upper().map(dialog_map).values
        return cohen_kappa_score(h, l, weights="quadratic")  # önerilen


# ------------------ ana hesaplama ------------------
def evaluate_all_kappas(df):
    pairs = {
        "human1": {
            "content": ("human1-content", "content_llm"),
            "dialog":  ("human1-dialog",  "dialog_llm"),
        },
        "human2": {
            "content": ("human2-content", "content_llm"),
            "dialog":  ("human2-dialog",  "dialog_llm"),
        },
        "human_main": {
            "content": ("content", "content_llm"),
            "dialog":  ("dialog",  "dialog_llm"),
        }
    }

    results = {}

    for human_name, dims in pairs.items():
        results[human_name] = {}
        for dim, (h_col, l_col) in dims.items():
            kappa = compute_kappa_pair(df, h_col, l_col, dim)
            results[human_name][dim] = kappa
            print(f"{human_name.upper()} - {dim.upper()} Kappa: "
                  f"{'N/A' if kappa is None else f'{kappa:.4f}'}")

    return results


# ------------------ çalıştır ------------------
all_results = evaluate_all_kappas(df)

In [ ]:
def analyze_errors(df: pd.DataFrame, top_n: int = 10):
    """Analyze prediction errors"""

    df['content_error'] = df['content'] != df['content_llm']
    df['dialog_error'] = df['dialog'].str.strip().str.upper() != df['dialog_llm'].str.strip().str.upper()

    content_errors = df[df['content_error']]
    dialog_errors = df[df['dialog_error']]

    print("\n" + "="*80)
    print("ERROR ANALYSIS")
    print("="*80)

    print(f"\nContent errors: {len(content_errors)} / {len(df)}")
    print(f"Dialog errors: {len(dialog_errors)} / {len(df)}")

    if len(content_errors) > 0:
        print(f"\n📊 Content Error Examples:")
        for i, (idx, row) in enumerate(content_errors.head(top_n).iterrows(), 1):
            print(f"\n{i}. {row['Message'][:80]}")
            print(f"   Human: {row['content']} | LLM: {row['content_llm']}")
            print(f"   {row['content_reasoning'][:80]}")

    if len(dialog_errors) > 0:
        print(f"\n\n📊 Dialog Error Examples:")
        for i, (idx, row) in enumerate(dialog_errors.head(top_n).iterrows(), 1):
            print(f"\n{i}. {row['Message'][:80]}")
            print(f"   Human: {row['dialog']} | LLM: {row['dialog_llm']}")
            print(f"   {row['dialog_reasoning'][:80]}")

analyze_errors(df, 5)

**FLEISS KAPPA**

In [ ]:
import numpy as np
import pandas as pd

# Eğer statsmodels varsa en kolayı bu:
try:
    from statsmodels.stats.inter_rater import fleiss_kappa
    _HAS_STATSMODELS = True
except Exception:
    _HAS_STATSMODELS = False

def _fleiss_kappa_manual(M: np.ndarray) -> float:
    """
    M: N x K count matrix (each row sums to n_raters)
    """
    N, K = M.shape
    n = M.sum(axis=1)[0]  # all rows same after filtering

    # p_j: overall proportion for category j
    p = M.sum(axis=0) / (N * n)

    # P_i: agreement for item i
    P = (np.sum(M * (M - 1), axis=1)) / (n * (n - 1))

    Pbar = P.mean()
    Pe = np.sum(p ** 2)

    if np.isclose(1 - Pe, 0):
        return np.nan
    return (Pbar - Pe) / (1 - Pe)

def _build_fleiss_matrix(df, cols, label_type):
    """
    cols: puanlayıcı kolonları listesi (örn. human1-content, human2-content, human-birlesik-content, content_llm)
    label_type: "content" or "dialog"
    return: (M, used_n_items)
    """
    if label_type == "content":
        categories = [0, 1, 2, 3]
        # tüm puanlayıcılar için geçerli satırları tut
        mask = pd.Series(True, index=df.index)
        for c in cols:
            mask &= df[c].apply(lambda x: (not pd.isna(x)) and str(x).strip().isdigit() and int(x) in categories)
        dff = df.loc[mask, cols].copy()
        if len(dff) == 0:
            return None, 0

        dff = dff.astype(int)
        # N x K count matrix
        M = np.zeros((len(dff), len(categories)), dtype=int)
        for i, row in enumerate(dff.values):
            for v in row:
                M[i, categories.index(v)] += 1
        return M, len(dff)

    else:
        categories = ['A', 'B', 'C', 'D']
        mask = pd.Series(True, index=df.index)
        for c in cols:
            mask &= df[c].apply(lambda x: (not pd.isna(x)) and str(x).strip().upper() in categories)
        dff = df.loc[mask, cols].copy()
        if len(dff) == 0:
            return None, 0

        dff = dff.astype(str).apply(lambda s: s.str.strip().str.upper())
        M = np.zeros((len(dff), len(categories)), dtype=int)
        for i, row in enumerate(dff.values):
            for v in row:
                M[i, categories.index(v)] += 1
        return M, len(dff)

def fleiss_kappa_for_all_raters(df):
    # "tüm puanlayıcılar" burada tanımlı: 3 insan + LLM
    content_cols = ["human1-content", "human2-content", "human-birlesik-content", "content", "content_llm"]
    dialog_cols  = ["human1-dialog",  "human2-dialog",  "human-birlesik-dialog",  "dialog", "dialog_llm"]

    results = {}

    # CONTENT
    M_content, n_content = _build_fleiss_matrix(df, content_cols, "content")
    if M_content is None:
        results["content_fleiss"] = None
        print("CONTENT Fleiss Kappa: N/A (geçerli ortak satır yok)")
    else:
        k = fleiss_kappa(M_content) if _HAS_STATSMODELS else _fleiss_kappa_manual(M_content)
        results["content_fleiss"] = float(k)
        print(f"CONTENT Fleiss Kappa: {k:.4f}  | items used: {n_content}  | raters: {len(content_cols)}")

    # DIALOG
    M_dialog, n_dialog = _build_fleiss_matrix(df, dialog_cols, "dialog")
    if M_dialog is None:
        results["dialog_fleiss"] = None
        print("DIALOG Fleiss Kappa: N/A (geçerli ortak satır yok)")
    else:
        k = fleiss_kappa(M_dialog) if _HAS_STATSMODELS else _fleiss_kappa_manual(M_dialog)
        results["dialog_fleiss"] = float(k)
        print(f"DIALOG Fleiss Kappa:  {k:.4f}  | items used: {n_dialog}  | raters: {len(dialog_cols)}")

    return results

# Çalıştır
fleiss_results = fleiss_kappa_for_all_raters(df)
